# Clase 3 (Parte 2): Construcción de una Arquitectura RAG desde Cero
**Prof. Leticia Rodriguez | Universidad de Buenos Aires**

Aunque los modelos modernos cuentan con ventanas de contexto masivas, procesar millones de documentos directamente en cada interacción genera costos computacionales elevados y latencia. Para solucionar esto de manera profesional, implementamos la arquitectura **RAG (Retrieval-Augmented Generation)**.

En esta sesión práctica construiremos el pipeline de recuperación completo paso a paso:
1. **Text Chunking:** Segmentación de nuestra base documental.
2. **Vectorization:** Generación de embeddings densos con el modelo de Google `text-embedding-004`.
3. **Análisis Matemático:** Comparación manual en Numpy de métricas de distancia (Euclidiana vs. Coseno).
4. **Vector Store Integration:** Indexación y búsqueda automatizada de alta velocidad utilizando **FAISS**.
5. **Grounding:** Inyección contextual estructurada para eliminar alucinaciones en la respuesta final.

Modelos de Google para Desarrolladores: https://ai.google.dev/gemini-api/docs/models

In [1]:
# Instalamos las dependencias core para el pipeline RAG y almacenamiento vectorial
!pip install -q google-generativeai faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.6 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import faiss
import google.genai as genai
from google.colab import userdata

# Inicialización del cliente unificado de desarrollo de Google GenAI
gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)

# Definimos de manera estricta los dos modelos a utilizar en nuestra arquitectura
MODELO_LLM = 'gemini-2.5-flash'
MODELO_EMBEDDING = 'gemini-embedding-2' # Modelo de representación vectorial nativo
print("Componentes de la arquitectura RAG inicializados con éxito.")

Componentes de la arquitectura RAG inicializados con éxito.


## PARTE 1: La Base de Conocimiento Corporativa (Text Chunking)
Simularemos el proceso de ingesta dividiendo normativas oficiales internas de la Universidad de Buenos Aires en fragmentos de texto semánticamente independientes llamados *chunks*.

In [3]:
# Inicializamos nuestra base documental segmentada
chunks_documento = [
    "Art 1: Las inscripciones a los exámenes finales de la Universidad de Buenos Aires cierran exactamente 48 horas hábiles antes de la fecha establecida para el inicio de la prueba.",
    "Art 2: La Maestría en Ciencias de Datos de la UBA exige un mínimo estricto del 80% de asistencia obligatoria presencial o sincrónica en cada módulo para mantener la condición de regularidad.",
    "Art 3: La biblioteca central ofrece préstamos de libros físicos por un período máximo de 7 días corridos, renovables una única vez por la plataforma web institucional.",
    "Art 4: Los trabajos finales de graduación de la Facultad de Ingeniería deben presentarse en formato digital PDF siguiendo estrictamente la plantilla institucional de la UBA."
]
print(f"Se han cargado y validado {len(chunks_documento)} fragmentos documentales para la indexación.")

Se han cargado y validado 4 fragmentos documentales para la indexación.


## PARTE 2: Generación de Embeddings Vectoriales Densos
Cada bloque de texto será enviado a la API de embeddings para transformarse en un vector numérico multidimensional que captura el significado conceptual y semántico subyacente.

In [4]:
print(f"Enviando fragmentos documentales al modelo {MODELO_EMBEDDING}...")
embeddings_chunks = []

for text in chunks_documento:
    respuesta_vectorial = client.models.embed_content(
        model=MODELO_EMBEDDING,
        contents=text
    )
    # Extraemos el vector abstracto de floats
    vector = respuesta_vectorial.embeddings[0].values
    embeddings_chunks.append(vector)

# Convertimos los vectores a una matriz estructurada de Numpy de precisión float32
matrix_embeddings = np.array(embeddings_chunks).astype('float32')
print(f" Matriz de almacenamiento vectorial construida. Dimensiones (Forma): {matrix_embeddings.shape}")
print(f"Cada uno de nuestros {matrix_embeddings.shape[0]} chunks está mapeado como un punto matemático en un espacio de {matrix_embeddings.shape[1]} dimensiones.")

Enviando fragmentos documentales al modelo gemini-embedding-2...
 Matriz de almacenamiento vectorial construida. Dimensiones (Forma): (4, 3072)
Cada uno de nuestros 4 chunks está mapeado como un punto matemático en un espacio de 3072 dimensiones.


## PARTE 3: Análisis de Álgebra Lineal en Ciencia de Datos (Numpy)
Para comprender a nivel algorítmico cómo opera el motor de búsqueda interno de un Vector Store, calcularemos de forma manual el coeficiente de distancia e intersección geométrica entre la pregunta de un alumno y nuestra base indexada.

In [5]:
query_alumno = "¿Cuántos días puedo tener prestado un libro físico de la biblioteca de la universidad?"

# Generamos de forma dinámica el embedding representativo para la consulta del usuario
query_embedding = np.array(
    client.models.embed_content(model=MODELO_EMBEDDING, contents=query_alumno).embeddings[0].values
).astype('float32')

print("--- CÁLCULO DE SIMILITUD GEOMÉTRICA CON NUMPY ---")
for idx, chunk_vector in enumerate(matrix_embeddings):
    # 1. Distancia Euclidiana (L2): Medición directa del segmento lineal entre dos puntos
    dist_l2 = np.linalg.norm(query_embedding - chunk_vector)

    # 2. Similitud de Coseno: Medición del ángulo angular entre ambos tensores
    dot_product = np.dot(query_embedding, chunk_vector)
    norm_query = np.linalg.norm(query_embedding)
    norm_chunk = np.linalg.norm(chunk_vector)
    sim_coseno = dot_product / (norm_query * norm_chunk)

    print(f"Chunk {idx+1} -> Distancia Euclidiana (Menor es mejor): {dist_l2:.4f} | Similitud Coseno (Mayor es mejor): {sim_coseno:.4f}")

--- CÁLCULO DE SIMILITUD GEOMÉTRICA CON NUMPY ---
Chunk 1 -> Distancia Euclidiana (Menor es mejor): 0.9099 | Similitud Coseno (Mayor es mejor): 0.5861
Chunk 2 -> Distancia Euclidiana (Menor es mejor): 0.9664 | Similitud Coseno (Mayor es mejor): 0.5330
Chunk 3 -> Distancia Euclidiana (Menor es mejor): 0.6801 | Similitud Coseno (Mayor es mejor): 0.7688
Chunk 4 -> Distancia Euclidiana (Menor es mejor): 0.9360 | Similitud Coseno (Mayor es mejor): 0.5619


## PARTE 4: Indexación Profesional con FAISS
En entornos de producción masivos, calcular distancias una por una de manera iterativa (KNN iterativo) es ineficiente. Utilizaremos **FAISS (Facebook AI Similarity Search)**, una librería optimizada para realizar búsquedas de vecinos cercanos a gran escala en microsegundos.

In [6]:
dimension_vectores = matrix_embeddings.shape[1]

print("Nuestros datos, los vectores, los textos sin hacer embedding")
print(chunks_documento)

print("Los embeddings de nuestro textos guardados en una Matriz")
print(matrix_embeddings)
# Clonamos y normalizamos los vectores matemáticos bajo la norma L2 para realizar Producto Escalar equivalente a Coseno
faiss_matrix = matrix_embeddings.copy()
faiss_query = query_embedding.copy().reshape(1, -1)
faiss.normalize_L2(faiss_matrix)
faiss.normalize_L2(faiss_query)

# Inicializamos un índice FAISS plano basado en Producto Escalar de alta eficiencia (IndexFlatIP)
index_store = faiss.IndexFlatIP(dimension_vectores)
index_store.add(faiss_matrix)

# Consultamos la base vectorial para extraer el elemento semánticamente más cercano (Top-1, k=1)
scores, indices = index_store.search(faiss_query, k=1)

id_documento_recuperado = indices[0][0]
contexto_factual = chunks_documento[id_documento_recuperado]

print("--- EXTRACCIÓN AUTOMATIZADA DESDE EL VECTOR STORE (FAISS) ---")
print(f"ID del fragmento óptimo recuperado: {id_documento_recuperado} (Score Coseno: {scores[0][0]:.4f})")
print(f"Contexto exacto recuperado: '{contexto_factual}'")

Nuestros datos, los vectores, los textos sin hacer embedding
['Art 1: Las inscripciones a los exámenes finales de la Universidad de Buenos Aires cierran exactamente 48 horas hábiles antes de la fecha establecida para el inicio de la prueba.', 'Art 2: La Maestría en Ciencias de Datos de la UBA exige un mínimo estricto del 80% de asistencia obligatoria presencial o sincrónica en cada módulo para mantener la condición de regularidad.', 'Art 3: La biblioteca central ofrece préstamos de libros físicos por un período máximo de 7 días corridos, renovables una única vez por la plataforma web institucional.', 'Art 4: Los trabajos finales de graduación de la Facultad de Ingeniería deben presentarse en formato digital PDF siguiendo estrictamente la plantilla institucional de la UBA.']
Los embeddings de nuestro textos guardados en una Matriz
[[-0.0141332  -0.02174101  0.00877076 ... -0.00876222  0.00502064
  -0.02615313]
 [-0.02495948 -0.01724009 -0.01906968 ... -0.00945892  0.01714452
  -0.002629

## PARTE 5: Grounding del Prompt y Ejecución del Modelo de Lenguaje
Finalmente, inyectamos el fragmento recuperado de forma estratégica dentro del prompt template para obligar a Gemini a basarse únicamente en hechos verdaderos, mitigando de raíz cualquier riesgo de alucinación informativa.

In [7]:
# Diseñamos la estructura del prompt final inyectando el contexto recuperado por FAISS
prompt_institucional_rag = f"""
Eres un asistente de regulaciones internas de la Universidad de Buenos Aires.
Responde la siguiente pregunta del usuario utilizando única, obligatoria y estrictamente la información provista dentro del bloque de contexto institucional.
Si la respuesta no puede deducirse claramente del contexto proveído, responde de manera formal indicando que la base de datos oficial actual no cuenta con información certera sobre el tema.

CONTEXTO INSTITUCIONAL EXCLUSIVO:
{contexto_factual}

PREGUNTA DEL USUARIO:
{query_alumno}

RESPUESTA BASADA EN EVIDENCIA:
"""

print("--- GENERANDO RESPUESTA CON AUMENTO DE CONTEXTO (RAG GROUNDING) ---")
respuesta_final_llm = client.models.generate_content(contents=prompt_institucional_rag, model=MODELO_LLM)
print(respuesta_final_llm.text)

--- GENERANDO RESPUESTA CON AUMENTO DE CONTEXTO (RAG GROUNDING) ---
Puedes tener prestado un libro físico por un período máximo de 7 días corridos. Este préstamo es renovable una única vez.
